In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import matplotlib.pyplot as plt

In [18]:
df_train = pd.read_csv("../data/processed/train_data.csv")
df_test = pd.read_csv("../data/processed/test_data.csv")

In [19]:
df_0 = df_train[~df_train["MonthlyIncome"].isna()].reset_index(drop=True)
df_1 = df_train[~df_train["NumberOfDependents"].isna()].reset_index(drop=True)

df_use_0 = df_0[[ 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse']]

df_use_1 = df_1[['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse']]


target_income = df_0['MonthlyIncome']
target_dependents = df_1['NumberOfDependents']

model_income = RandomForestRegressor()
model_income.fit(df_use_0, target_income)

model_dependents = RandomForestRegressor()
model_dependents.fit(df_use_1, target_dependents)

KeyboardInterrupt: 

In [ ]:
def retornaValorIncome(x):
    X = x[['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse']]
    return model_income.predict(X)

def retornaValorDependents(x):
    X = x[['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse']]
    return model_dependents.predict(X)

In [ ]:
mask_income = df_train["MonthlyIncome"].isnull()
df_train.loc[mask_income, "MonthlyIncome"] = retornaValorIncome(df_train[mask_income])

mask_dep = df_train["NumberOfDependents"].isnull()
df_train.loc[mask_dep, "NumberOfDependents"] = retornaValorDependents(df_train[mask_dep])

mask_income = df_test["MonthlyIncome"].isnull()
df_test.loc[mask_income, "MonthlyIncome"] = retornaValorIncome(df_test[mask_income])

mask_dep = df_test["NumberOfDependents"].isnull()
df_test.loc[mask_dep, "NumberOfDependents"] = retornaValorDependents(df_test[mask_dep])

In [ ]:
column_mapping = {'SeriousDlqin2yrs': 'Target', 'RevolvingUtilizationOfUnsecuredLines': 'credit_utilization', 'age': 'age', 
                  'NumberOfTime30-59DaysPastDueNotWorse': 'late_30_59_days', 'DebtRatio': 'debt_ratio', 'MonthlyIncome': 'monthly_income', 
                  'NumberOfOpenCreditLinesAndLoans': 'open_credit_lines', 'NumberOfTimes90DaysLate': 'late_90_days', 
                  'NumberRealEstateLoansOrLines': 'real_estate_loans', 'NumberOfTime60-89DaysPastDueNotWorse': 'late_60_89_days', 
                  'NumberOfDependents': 'dependents'}

df_train = df_train.rename(columns=column_mapping)

In [ ]:
cols_clip = ['credit_utilization','late_30_59_days', 'debt_ratio', 'monthly_income', 'open_credit_lines', 'late_90_days','real_estate_loans', 'late_60_89_days', 'dependents']

In [ ]:
for col in cols_clip:
    lower = df_train[col].quantile(0.01)
    upper = df_train[col].quantile(0.99)
    df_train[col] = df_train[col].clip(lower=lower, upper=upper)

In [ ]:
df_train["income_per_dependent"] = df_train["monthly_income"] / (df_train["dependents"] + 1)
df_train["weighted_late"] = ((1 * df_train['late_30_59_days']) + (2 * df_train['late_60_89_days']) + (3 * df_train['late_90_days']))

In [ ]:
df_train

,Target,credit_utilization,age,late_30_59_days,debt_ratio,monthly_income,open_credit_lines,late_90_days,real_estate_loans,late_60_89_days,dependents,income_per_dependent,weighted_late
0,1,0.766127,45,2,0.802982,9120.0,13,0,4,0,2.0,3040.000000,2
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0,1300.000000,0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0,3042.000000,4
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0,3300.000000,0
4,0,0.907239,49,1,0.024926,23250.0,7,0,1,0,0.0,23250.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,0,0.040674,74,0,0.225131,2100.0,4,0,1,0,0.0,2100.000000,0
149996,0,0.299745,44,0,0.716562,5584.0,4,0,1,0,2.0,1861.333333,0
149997,0,0.246044,58,0,3870.000000,0.0,18,0,1,0,0.0,0.000000,0
149998,0,0.000000,30,0,0.000000,5716.0,4,0,0,0,0.0,5716.000000,0


In [ ]:
df_X = df_train.drop("Target",axis=1)
df_y = df_train[["Target"]]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_X,df_y, test_size=0.33,stratify=df_y,random_state=42)

In [ ]:
X_train.to_parquet("../data/processed/X_train.parquet") 
X_test.to_parquet("../data/processed/X_test.parquet")
y_train.to_parquet("../data/processed/y_train.parquet")
y_test.to_parquet("../data/processed/y_test.parquet") 